# 🚀 Boosting — Sequential Ensemble Learning

> **Folder:** `08_Ensemble_Learning`  
> **Notebook:** `boosting.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Understand **how boosting reduces bias** by sequentially correcting errors
- Implement **AdaBoost** — the original boosting algorithm
- Use **Gradient Boosting** — the most widely used boosting method
- Compare **AdaBoost vs GBM vs XGBoost vs LightGBM**
- Tune key boosting hyperparameters: `n_estimators`, `learning_rate`, `max_depth`
- Use **early stopping** to prevent overfitting
- Visualize the **staged prediction** — how boosting improves iteration by iteration
- Understand the **bias-variance** perspective of boosting

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Classification + Regression |
| 2 | AdaBoost — Classification | Reweights misclassified samples |
| 3 | AdaBoost — Staged Predictions | Per-iteration improvement |
| 4 | Gradient Boosting (sklearn) | Fits residuals sequentially |
| 5 | Learning Rate vs n_estimators | Shrinkage tradeoff |
| 6 | max_depth Sensitivity | Shallow trees = weak learners |
| 7 | Subsample — Stochastic GBM | Variance reduction in boosting |
| 8 | Early Stopping | Prevent overfitting automatically |
| 9 | XGBoost Tuning | Regularized boosting |
| 10 | LightGBM Tuning | Leaf-wise growth, fastest |
| 11 | Boosting vs Bagging | Sequential vs parallel |
| 12 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    BaggingClassifier, RandomForestClassifier,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    r2_score, mean_squared_error,
)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed — pip install xgboost")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("LightGBM not installed — pip install lightgbm")

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded!')

---
## 1️⃣ Dataset Setup

In [ ]:
np.random.seed(42)

X_clf, y_clf = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2, weights=[0.5, 0.5], random_state=42
)
X_clf_df = pd.DataFrame(X_clf, columns=[f'F{i+1:02d}' for i in range(20)])
y_clf_s  = pd.Series(y_clf, name='Target')

X_reg, y_reg = make_regression(
    n_samples=800, n_features=15, n_informative=8, noise=25, random_state=42
)
X_reg_df = pd.DataFrame(X_reg, columns=[f'R{i+1:02d}' for i in range(15)])
y_reg_s  = pd.Series(y_reg, name='Target')

sc_clf = StandardScaler(); sc_reg = StandardScaler()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf_df, y_clf_s, test_size=0.2, stratify=y_clf_s, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_df, y_reg_s, test_size=0.2, random_state=42)
Xc_tr_sc = sc_clf.fit_transform(Xc_tr); Xc_te_sc = sc_clf.transform(Xc_te)
Xr_tr_sc = sc_reg.fit_transform(Xr_tr); Xr_te_sc = sc_reg.transform(Xr_te)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf  = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Classification: {X_clf_df.shape} | classes={dict(y_clf_s.value_counts().sort_index())}')
print(f'  Train={Xc_tr.shape}  Test={Xc_te.shape}')
print(f'Regression    : {X_reg_df.shape}')

---
## 2️⃣ AdaBoost — Adaptive Boosting

> **AdaBoost** trains estimators sequentially, each focusing on the samples  
> the previous estimator got **wrong** by increasing their weights.
>
> ```
> Algorithm:
>   Initialize: equal weights wᵢ = 1/n for each sample
>   For t = 1, 2, ..., T:
>     1. Train weak learner hₜ on weighted samples
>     2. Compute weighted error: εₜ = Σ wᵢ × 1[hₜ(xᵢ) ≠ yᵢ]
>     3. Compute estimator weight: αₜ = 0.5 × log((1−εₜ)/εₜ)
>     4. Update sample weights:
>          wᵢ ← wᵢ × exp(−αₜ × yᵢ × hₜ(xᵢ))
>          Normalize so Σwᵢ = 1
>   Final: H(x) = sign(Σ αₜ × hₜ(x))
> ```
>
> ⚠️ AdaBoost is sensitive to **outliers** — noisy samples get very high weights.


In [ ]:
# ── AdaBoost Classification ───────────────────────────────────────────────
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learner = stump
    n_estimators=200,
    learning_rate=1.0,
    algorithm='SAMME',
    random_state=42,
)
ada_clf.fit(Xc_tr_sc, yc_tr)

ada_tr_auc = roc_auc_score(yc_tr, ada_clf.predict_proba(Xc_tr_sc)[:,1])
ada_te_auc = roc_auc_score(yc_te, ada_clf.predict_proba(Xc_te_sc)[:,1])
ada_te_acc = accuracy_score(yc_te, ada_clf.predict(Xc_te_sc))

print('AdaBoost Classifier (200 stumps):')
print(f'  Train AUC : {ada_tr_auc:.4f}')
print(f'  Test AUC  : {ada_te_auc:.4f}')
print(f'  Test Acc  : {ada_te_acc:.4f}')
print(f'  Overfit   : {ada_tr_auc - ada_te_auc:.4f}')

# Staged test error
staged_tr = [roc_auc_score(yc_tr, p[:,1])
             for p in ada_clf.staged_predict_proba(Xc_tr_sc)]
staged_te = [roc_auc_score(yc_te, p[:,1])
             for p in ada_clf.staged_predict_proba(Xc_te_sc)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
iters = range(1, len(staged_tr)+1)
axes[0].plot(iters, staged_tr, color=COLORS['primary'],
             linewidth=2, label='Train AUC')
axes[0].plot(iters, staged_te, color=COLORS['secondary'],
             linewidth=2, label='Test AUC')
axes[0].axvline(np.argmax(staged_te)+1, color=COLORS['accent'],
                linestyle='--', linewidth=2,
                label=f'Best iter={np.argmax(staged_te)+1}')
axes[0].set_xlabel('Number of Estimators', fontsize=11)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('AdaBoost — Staged AUC (Train vs Test)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Estimator weights (alpha values)
axes[1].plot(range(1, len(ada_clf.estimator_weights_)+1),
             ada_clf.estimator_weights_,
             'o-', color=COLORS['purple'], linewidth=1.5, markersize=3)
axes[1].set_xlabel('Estimator Index', fontsize=11)
axes[1].set_ylabel('Estimator Weight (αₜ)', fontsize=11)
axes[1].set_title('AdaBoost — Estimator Weights
(higher α = more accurate estimator)',
                  fontsize=12, fontweight='bold')

plt.suptitle('AdaBoost — Adaptive Boosting', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3️⃣ AdaBoost — Base Estimator Depth Sensitivity

> AdaBoost traditionally uses **decision stumps** (max_depth=1) as weak learners.  
> Deeper trees make each estimator stronger but increase the risk of overfitting.


In [ ]:
depth_values = [1, 2, 3, 5]
ada_depth_results = []

fig, ax = plt.subplots(figsize=(13, 5))

for depth, color in zip(depth_values, COLORS['palette'][:4]):
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=depth),
        n_estimators=200, learning_rate=1.0,
        algorithm='SAMME', random_state=42,
    )
    ada.fit(Xc_tr_sc, yc_tr)
    staged = [roc_auc_score(yc_te, p[:,1])
              for p in ada.staged_predict_proba(Xc_te_sc)]
    best_auc = max(staged)
    best_it  = np.argmax(staged) + 1
    ada_depth_results.append({
        'max_depth': depth, 'Best AUC': round(best_auc,4),
        'Best Iter': best_it
    })
    ax.plot(range(1, 201), staged, color=color, linewidth=2,
            label=f'depth={depth} (best={best_auc:.4f} @{best_it})')

ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('Test ROC-AUC', fontsize=11)
ax.set_title('AdaBoost — Base Estimator Depth Sensitivity',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

print(pd.DataFrame(ada_depth_results).to_string(index=False))

---
## 4️⃣ Gradient Boosting — Fitting Residuals Sequentially

> Gradient Boosting fits each new tree to the **negative gradient** of the loss —  
> which for MSE loss is simply the **residuals**: actual − predicted.
>
> ```
> Initialize: F₀(x) = mean(y)
> For t = 1, 2, ..., T:
>   1. Compute pseudo-residuals: rᵢₜ = −∂L(yᵢ, Fₜ₋₁(xᵢ)) / ∂Fₜ₋₁(xᵢ)
>      (for MSE: rᵢₜ = yᵢ − Fₜ₋₁(xᵢ))
>   2. Fit a tree hₜ to the pseudo-residuals
>   3. Update: Fₜ(x) = Fₜ₋₁(x) + η × hₜ(x)
>      (η = learning_rate = shrinkage factor)
> Final: F_T(x)
> ```
>
> Unlike AdaBoost, GBM works for **any differentiable loss function**.


In [ ]:
# ── GBM Classification ────────────────────────────────────────────────────
gbm_clf = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=1.0,
    min_samples_leaf=1,
    random_state=42,
)
gbm_clf.fit(Xc_tr_sc, yc_tr)

gbm_tr_auc = roc_auc_score(yc_tr, gbm_clf.predict_proba(Xc_tr_sc)[:,1])
gbm_te_auc = roc_auc_score(yc_te, gbm_clf.predict_proba(Xc_te_sc)[:,1])

# Staged scores
gbm_staged_tr = [roc_auc_score(yc_tr, p[:,1])
                 for p in gbm_clf.staged_predict_proba(Xc_tr_sc)]
gbm_staged_te = [roc_auc_score(yc_te, p[:,1])
                 for p in gbm_clf.staged_predict_proba(Xc_te_sc)]

best_iter_gbm = np.argmax(gbm_staged_te) + 1
print(f'GradientBoosting (n=300, lr=0.05, depth=3):')
print(f'  Train AUC  : {gbm_tr_auc:.4f}')
print(f'  Test AUC   : {gbm_te_auc:.4f}')
print(f'  Best iter  : {best_iter_gbm} (AUC={max(gbm_staged_te):.4f})')
print(f'  Overfit    : {gbm_tr_auc - gbm_te_auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
iters_gbm = range(1, 301)
axes[0].plot(iters_gbm, gbm_staged_tr, color=COLORS['primary'],
             linewidth=2, label='Train AUC')
axes[0].plot(iters_gbm, gbm_staged_te, color=COLORS['secondary'],
             linewidth=2, label='Test AUC')
axes[0].axvline(best_iter_gbm, color=COLORS['accent'], linestyle='--',
                linewidth=2, label=f'Best iter={best_iter_gbm}')
axes[0].fill_between(iters_gbm, gbm_staged_tr, gbm_staged_te,
                     alpha=0.10, color=COLORS['warning'],
                     label='Overfit region')
axes[0].set_xlabel('n_estimators', fontsize=11)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('GBM — Staged AUC (Train vs Test)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# Feature importance
feat_imp = pd.Series(gbm_clf.feature_importances_,
                      index=Xc_tr.columns).sort_values(ascending=False)
axes[1].barh(feat_imp.index[:15], feat_imp.values[:15],
             color=COLORS['primary'], alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Feature Importance (MDI)', fontsize=11)
axes[1].set_title('GBM — Top 15 Feature Importances',
                  fontsize=12, fontweight='bold')

plt.suptitle('Gradient Boosting Classifier', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5️⃣ Learning Rate vs n_estimators — The Shrinkage Tradeoff

> **Learning rate (η)** shrinks the contribution of each tree:
>
> `Fₜ(x) = Fₜ₋₁(x) + η × hₜ(x)`
>
> | Learning Rate | n_estimators needed | Generalization |
> |:-------------:|:-------------------:|:--------------:|
> | High (0.3+) | Few (50–100) | ⚠️ Risky — may overfit |
> | Medium (0.05–0.1) | Medium (200–500) | ✅ Good balance |
> | Low (0.01–0.05) | Many (500–2000) | ✅ Best generalization |
>
> **Rule:** Lower learning rate + more trees = better generalization (if compute allows).


In [ ]:
lr_configs = [
    (0.001, 500, COLORS['purple']),
    (0.01,  400, COLORS['primary']),
    (0.05,  300, COLORS['accent']),
    (0.1,   200, COLORS['warning']),
    (0.3,   100, COLORS['secondary']),
]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
lr_results = []

for lr, n_est, color in lr_configs:
    gbm = GradientBoostingClassifier(
        n_estimators=n_est, learning_rate=lr,
        max_depth=3, random_state=42
    )
    gbm.fit(Xc_tr_sc, yc_tr)
    staged_te = [roc_auc_score(yc_te, p[:,1])
                 for p in gbm.staged_predict_proba(Xc_te_sc)]
    staged_tr = [roc_auc_score(yc_tr, p[:,1])
                 for p in gbm.staged_predict_proba(Xc_tr_sc)]

    best_auc = max(staged_te)
    best_it  = np.argmax(staged_te) + 1
    lr_results.append({
        'lr': lr, 'n_est': n_est,
        'Best Test AUC': round(best_auc, 4),
        'Best Iter': best_it,
        'Final Train AUC': round(staged_tr[-1], 4),
    })
    axes[0].plot(range(1, n_est+1), staged_te, color=color, linewidth=2,
                 label=f'lr={lr}, n={n_est} (best={best_auc:.4f})')

axes[0].set_xlabel('n_estimators', fontsize=11)
axes[0].set_ylabel('Test ROC-AUC', fontsize=11)
axes[0].set_title('Learning Rate vs n_estimators Tradeoff',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8)

lr_df = pd.DataFrame(lr_results)
print('Learning Rate Sensitivity:')
print(lr_df.to_string(index=False))

bars = axes[1].bar([str(r['lr']) for r in lr_results],
                    [r['Best Test AUC'] for r in lr_results],
                    color=[c for _, _, c in lr_configs], alpha=0.85,
                    edgecolor='white')
for bar, val in zip(bars, [r['Best Test AUC'] for r in lr_results]):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.001,
                 f'{val:.4f}', ha='center', fontsize=10)
axes[1].set_xlabel('Learning Rate (η)', fontsize=11)
axes[1].set_ylabel('Best Test ROC-AUC', fontsize=11)
axes[1].set_title('Best AUC per Learning Rate', fontsize=12, fontweight='bold')
axes[1].set_ylim([0.8, 1.0])

plt.suptitle('Learning Rate — Shrinkage Tradeoff', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6️⃣ max_depth — Weak vs Strong Learners

> GBM uses **shallow trees** (weak learners) to prevent overfitting:
> - `max_depth=1` → stumps (additive model, no interactions)
> - `max_depth=3` → default, captures 3-way interactions
> - `max_depth=5+` → stronger learners, more overfit risk
>
> Boosting works best with **weak learners** (max_depth = 2–5).  
> Contrast with bagging which works best with deep, overfit trees.


In [ ]:
depth_range  = [1, 2, 3, 4, 5, 6, 8]
depth_results = []

for d in depth_range:
    gbm = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=d, random_state=42
    )
    gbm.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, gbm.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, gbm.predict_proba(Xc_te_sc)[:,1])
    depth_results.append({
        'max_depth': d,
        'Train AUC': round(tr_auc, 4),
        'Test AUC' : round(te_auc, 4),
        'Overfit'  : round(tr_auc - te_auc, 4),
    })

depth_df = pd.DataFrame(depth_results)
print('max_depth Sensitivity (n=200, lr=0.05):')
print(depth_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(depth_df['max_depth'], depth_df['Train AUC'], 'o-',
             color=COLORS['primary'], linewidth=2.5, markersize=9,
             label='Train AUC')
axes[0].plot(depth_df['max_depth'], depth_df['Test AUC'], 's-',
             color=COLORS['secondary'], linewidth=2.5, markersize=9,
             label='Test AUC')
for _, row in depth_df.iterrows():
    axes[0].annotate(f'{row["Test AUC"]:.4f}',
                     (row['max_depth'], row['Test AUC']),
                     textcoords='offset points', xytext=(5,5), fontsize=8)
axes[0].set_xlabel('max_depth', fontsize=11)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('GBM — max_depth Sensitivity', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

gap_colors = [COLORS['secondary'] if g > 0.05 else COLORS['accent']
              for g in depth_df['Overfit']]
axes[1].bar(depth_df['max_depth'].astype(str), depth_df['Overfit'],
            color=gap_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0.05, color=COLORS['warning'], linestyle='--',
                linewidth=2, label='Overfit threshold (0.05)')
axes[1].set_xlabel('max_depth', fontsize=11)
axes[1].set_ylabel('Train − Test AUC', fontsize=11)
axes[1].set_title('Overfitting Gap vs max_depth', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('GBM — max_depth: Weak vs Strong Learners',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7️⃣ Subsample — Stochastic Gradient Boosting

> Adding **row subsampling** (`subsample < 1.0`) makes GBM stochastic:
> - Each tree is trained on a random fraction of training data
> - Introduces randomness → reduces correlation between trees → reduces variance
> - Often improves generalization (similar benefit to bagging)
>
> `subsample=0.8` is a good default for stochastic GBM.


In [ ]:
subsample_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sub_results = []

for ss in subsample_values:
    gbm = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=3, subsample=ss, random_state=42
    )
    gbm.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, gbm.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, gbm.predict_proba(Xc_te_sc)[:,1])
    sub_results.append({
        'subsample': ss,
        'Train AUC': round(tr_auc, 4),
        'Test AUC' : round(te_auc, 4),
        'Overfit'  : round(tr_auc - te_auc, 4),
    })

sub_df = pd.DataFrame(sub_results)
print('Subsample Sensitivity (n=200, lr=0.05, depth=3):')
print(sub_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(sub_df['subsample'], sub_df['Train AUC'], 'o-',
        color=COLORS['primary'], linewidth=2.5, markersize=8,
        label='Train AUC')
ax.plot(sub_df['subsample'], sub_df['Test AUC'], 's-',
        color=COLORS['secondary'], linewidth=2.5, markersize=8,
        label='Test AUC')
best_sub = sub_df.loc[sub_df['Test AUC'].idxmax(), 'subsample']
ax.axvline(best_sub, color=COLORS['accent'], linestyle='--',
           linewidth=2, label=f'Best subsample={best_sub}')
ax.set_xlabel('subsample', fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('Stochastic GBM — subsample Sensitivity
'
             '(subsample < 1.0 → Stochastic GBM)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

---
## 8️⃣ Early Stopping — Prevent Overfitting Automatically

> Rather than fixing `n_estimators` upfront, **early stopping** monitors  
> validation performance and stops when it stops improving.
>
> sklearn GBM: use `n_iter_no_change` + `validation_fraction`  
> XGBoost/LightGBM: use `early_stopping_rounds` with eval_set


In [ ]:
# ── sklearn GBM early stopping ────────────────────────────────────────────
gbm_es = GradientBoostingClassifier(
    n_estimators=1000,          # upper bound
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    validation_fraction=0.15,   # 15% of train for early stopping
    n_iter_no_change=20,        # stop if no improvement for 20 rounds
    tol=1e-4,
    random_state=42,
)
gbm_es.fit(Xc_tr_sc, yc_tr)

n_actual = gbm_es.n_estimators_
te_auc   = roc_auc_score(yc_te, gbm_es.predict_proba(Xc_te_sc)[:,1])
print(f'GBM with Early Stopping:')
print(f'  Requested n_estimators : 1000')
print(f'  Actual trees built     : {n_actual}  ← stopped early!')
print(f'  Test AUC               : {te_auc:.4f}')
print(f'  Compute saved          : {(1000-n_actual)/1000*100:.1f}%')

# Staged curve to visualize stopping point
staged_te_es = [roc_auc_score(yc_te, p[:,1])
                for p in gbm_es.staged_predict_proba(Xc_te_sc)]
staged_tr_es = [roc_auc_score(yc_tr, p[:,1])
                for p in gbm_es.staged_predict_proba(Xc_tr_sc)]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(1, n_actual+1), staged_tr_es, color=COLORS['primary'],
        linewidth=2, label='Train AUC')
ax.plot(range(1, n_actual+1), staged_te_es, color=COLORS['secondary'],
        linewidth=2, label='Val/Test AUC')
ax.axvline(n_actual, color=COLORS['accent'], linestyle='--', linewidth=2.5,
           label=f'Early stop @ iter {n_actual}')
ax.axvline(n_actual - 20, color=COLORS['warning'], linestyle=':',
           linewidth=2, label=f'Best iter @ {np.argmax(staged_te_es)+1}')
ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title(f'GBM Early Stopping — Stopped at iter {n_actual} of 1000',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

---
## 9️⃣ XGBoost — Regularized Gradient Boosting

> XGBoost adds **L1 and L2 regularization** directly into the tree-building  
> objective — making it more robust to overfitting than standard GBM.
>
> Key XGBoost-specific parameters:
> - `reg_alpha` — L1 regularization on leaf weights (sparsity)
> - `reg_lambda` — L2 regularization on leaf weights (smoothing)
> - `min_child_weight` — minimum sum of hessian in a leaf (controls tree depth)
> - `colsample_bytree` — fraction of features per tree (like RF)
> - `tree_method='hist'` — histogram-based splits (fast!)


In [ ]:
if XGB_AVAILABLE:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=3,
        tree_method='hist',
        eval_metric='auc',
        early_stopping_rounds=30,
        random_state=42,
        verbosity=0,
    )
    xgb_clf.fit(
        Xc_tr_sc, yc_tr,
        eval_set=[(Xc_tr_sc, yc_tr), (Xc_te_sc, yc_te)],
        verbose=False,
    )

    xgb_best_iter = xgb_clf.best_iteration
    xgb_te_auc    = roc_auc_score(yc_te, xgb_clf.predict_proba(Xc_te_sc)[:,1])
    xgb_tr_auc    = roc_auc_score(yc_tr, xgb_clf.predict_proba(Xc_tr_sc)[:,1])

    print(f'XGBoost:')
    print(f'  Best iteration : {xgb_best_iter}')
    print(f'  Train AUC      : {xgb_tr_auc:.4f}')
    print(f'  Test AUC       : {xgb_te_auc:.4f}')
    print(f'  Overfit gap    : {xgb_tr_auc - xgb_te_auc:.4f}')

    # XGBoost feature importance
    xgb_imp = pd.Series(xgb_clf.feature_importances_,
                         index=Xc_tr.columns).sort_values(ascending=False)

    # Training history
    results = xgb_clf.evals_result()
    tr_auc_hist = results['validation_0']['auc']
    te_auc_hist = results['validation_1']['auc']

    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    axes[0].plot(tr_auc_hist, color=COLORS['primary'],
                 linewidth=2, label='Train AUC')
    axes[0].plot(te_auc_hist, color=COLORS['secondary'],
                 linewidth=2, label='Val AUC')
    axes[0].axvline(xgb_best_iter, color=COLORS['accent'],
                    linestyle='--', linewidth=2,
                    label=f'Best iter={xgb_best_iter}')
    axes[0].set_xlabel('Iteration', fontsize=11)
    axes[0].set_ylabel('AUC', fontsize=11)
    axes[0].set_title('XGBoost — Training History (with early stopping)',
                      fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10)

    axes[1].barh(xgb_imp.index[:15], xgb_imp.values[:15],
                 color=COLORS['warning'], alpha=0.85, edgecolor='white')
    axes[1].set_xlabel('Feature Importance', fontsize=11)
    axes[1].set_title('XGBoost — Top 15 Feature Importances',
                      fontsize=12, fontweight='bold')

    plt.suptitle('XGBoost — Regularized Gradient Boosting',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('XGBoost not available — pip install xgboost')

---
## 🔟 LightGBM — Leaf-Wise Growth, Fastest Boosting

> LightGBM grows trees **leaf-wise** (best-first) rather than level-wise:
> - Finds the leaf with the maximum gain and splits it
> - Achieves lower loss with fewer leaves than level-wise growth
> - Much faster than XGBoost for large datasets
>
> Key LightGBM-specific parameter:
> - `num_leaves` — max leaves per tree (primary complexity control, NOT max_depth)
> - `min_child_samples` — minimum samples per leaf (regularization)


In [ ]:
if LGB_AVAILABLE:
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=20,
        random_state=42,
        verbose=-1,
    )

    lgb_clf.fit(
        Xc_tr_sc, yc_tr,
        eval_set=[(Xc_te_sc, yc_te)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    lgb_te_auc = roc_auc_score(yc_te, lgb_clf.predict_proba(Xc_te_sc)[:,1])
    lgb_tr_auc = roc_auc_score(yc_tr, lgb_clf.predict_proba(Xc_tr_sc)[:,1])

    print(f'LightGBM:')
    print(f'  Best iteration : {lgb_clf.best_iteration_}')
    print(f'  Train AUC      : {lgb_tr_auc:.4f}')
    print(f'  Test AUC       : {lgb_te_auc:.4f}')
    print(f'  Overfit gap    : {lgb_tr_auc - lgb_te_auc:.4f}')

    lgb_imp = pd.Series(lgb_clf.feature_importances_,
                         index=Xc_tr.columns).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    evals_result = lgb_clf.evals_result_
    te_hist = evals_result['valid_0']['binary_logloss']
    axes[0].plot(te_hist, color=COLORS['primary'], linewidth=2,
                 label='Val Log Loss')
    axes[0].axvline(lgb_clf.best_iteration_, color=COLORS['secondary'],
                    linestyle='--', linewidth=2,
                    label=f'Best iter={lgb_clf.best_iteration_}')
    axes[0].set_xlabel('Iteration', fontsize=11)
    axes[0].set_ylabel('Log Loss', fontsize=11)
    axes[0].set_title('LightGBM — Val Loss History', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10)

    axes[1].barh(lgb_imp.index[:15], lgb_imp.values[:15],
                 color=COLORS['accent'], alpha=0.85, edgecolor='white')
    axes[1].set_xlabel('Feature Importance (split count)', fontsize=11)
    axes[1].set_title('LightGBM — Top 15 Feature Importances',
                      fontsize=12, fontweight='bold')

    plt.suptitle('LightGBM — Leaf-Wise Gradient Boosting',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('LightGBM not available — pip install lightgbm')

---
## 1️⃣1️⃣ Boosting vs Bagging — Full Comparison

> | Aspect | Bagging | Boosting |
> |--------|:-------:|:--------:|
> | Order | Parallel | Sequential |
> | Reduces | Variance | Bias |
> | Base estimator | High-variance (deep tree) | Low-variance (shallow tree) |
> | Sensitive to outliers | Low | High (AdaBoost) |
> | Risk of overfitting | Low | Medium (needs early stopping) |
> | Speed | ✅ Parallelizable | ❌ Sequential |
> | Best for | High-variance models | High-bias models |


In [ ]:
comparison_models = {
    'Single Tree (deep)'   : DecisionTreeClassifier(max_depth=None, random_state=42),
    'Single Tree (d=3)'    : DecisionTreeClassifier(max_depth=3,    random_state=42),
    'BaggingClassifier'    : BaggingClassifier(
        DecisionTreeClassifier(max_depth=None),
        n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest'        : RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost'             : AdaBoostClassifier(
        DecisionTreeClassifier(max_depth=1),
        n_estimators=200, learning_rate=0.5,
        algorithm='SAMME', random_state=42),
    'GradientBoosting'     : GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=3, subsample=0.8, random_state=42),
}

if XGB_AVAILABLE:
    comparison_models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0, tree_method='hist')
if LGB_AVAILABLE:
    comparison_models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05,
        num_leaves=31, subsample=0.8,
        random_state=42, verbose=-1)

comp_rows = []
for name, model in comparison_models.items():
    model.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, model.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, model.predict_proba(Xc_te_sc)[:,1])
    te_acc = accuracy_score(yc_te, model.predict(Xc_te_sc))
    comp_rows.append({
        'Model'     : name,
        'Train AUC' : round(tr_auc, 4),
        'Test AUC'  : round(te_auc, 4),
        'Test Acc'  : round(te_acc, 4),
        'Overfit'   : round(tr_auc - te_auc, 4),
    })
    print(f'  {name:25s}: Train={tr_auc:.4f} | Test={te_auc:.4f}')

comp_df = pd.DataFrame(comp_rows).sort_values('Test AUC', ascending=False)
print('
Full Comparison:')
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
x = np.arange(len(comp_df)); w = 0.35
axes[0].bar(x-w/2, comp_df['Train AUC'], w, label='Train AUC',
            color=COLORS['primary'], alpha=0.85)
axes[0].bar(x+w/2, comp_df['Test AUC'],  w, label='Test AUC',
            color=COLORS['accent'], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comp_df['Model'], rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('All Models — Train vs Test AUC', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].set_ylim([0.6, 1.05])

gap_colors = [COLORS['secondary'] if g > 0.05 else COLORS['accent']
              for g in comp_df['Overfit']]
axes[1].barh(comp_df['Model'], comp_df['Overfit'],
             color=gap_colors, alpha=0.85, edgecolor='white')
axes[1].axvline(0.05, color=COLORS['warning'], linestyle='--',
                linewidth=2, label='Overfit threshold')
axes[1].set_xlabel('Train − Test AUC', fontsize=11)
axes[1].set_title('Overfitting Gap per Model', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Boosting vs Bagging — Complete Leaderboard',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## ✅ 12. Summary & Golden Rules

| Algorithm | Reduces | Base Learner | Outlier Sensitivity | Speed |
|-----------|:-------:|:------------:|:-------------------:|:-----:|
| **AdaBoost** | Bias | Stumps (depth=1) | ❌ High | ✅ Fast |
| **GBM (sklearn)** | Bias + some variance | Shallow trees (depth=3) | ⚠️ Medium | ⚠️ Medium |
| **XGBoost** | Bias + variance (L1/L2) | Shallow trees | ⚠️ Medium | ✅ Fast (hist) |
| **LightGBM** | Bias + variance | Leaf-wise | ⚠️ Medium | ✅ Fastest |
| **Bagging/RF** | Variance | Deep trees | ✅ Low | ✅ Parallel |

### 🔑 Golden Rules

1. **Boosting reduces bias** — ideal when the base model underfits
2. **Bagging reduces variance** — ideal when the base model overfits
3. **Always use early stopping** — prevents overfitting without manual n_estimators tuning
4. **Lower learning rate + more trees** = better generalization (if compute allows)
5. **max_depth=3–5** is the sweet spot for GBM — weak learners, not stumps
6. **subsample=0.8** adds useful stochasticity — reduces variance in GBM
7. **AdaBoost is sensitive to outliers** — clean data before using it
8. **XGBoost reg_alpha + reg_lambda** provide additional overfit protection
9. **LightGBM num_leaves is the primary complexity control** — not max_depth
10. **Feature importance from GBM/XGB is MDI-based** — complement with permutation importance

---

## 🔗 Next Steps

- ➡️ `08_Ensemble_Learning/stacking.ipynb` — Meta-learning ensemble
- ➡️ `08_Ensemble_Learning/bagging.ipynb` — Parallel ensemble (variance reduction)
- ➡️ `07_Hyperparameter_Tuning/randomsearchcv.ipynb` — Tune boosting hyperparameters
- ➡️ `05_Model_Evaluation/bias_variance_tradeoff.md` — Why boosting reduces bias
